In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Data Reading

In [0]:
df = spark.read.format("parquet").load("abfss://bronzestg@storageloweretep1.dfs.core.windows.net/buyer")

In [0]:
df=df.drop("_rescued_data")

_fetching domains_

In [0]:
df=df.withColumn("domains",split(col("email"),"@")[1])
df.display()

In [0]:
df.groupBy(col("domains")).agg(count("customer_id").alias("total_buyer")).sort("total_buyer",ascending=False).display()

In [0]:
df_gmail = df.filter(col("domains")=="gmail.com")
df_yahoo = df.filter(col("domains")=="yahoo.com")
df_hotMail = df.filter(col("domains")=="hotmail.com")

In [0]:
df = df.withColumn("full_name",concat(col("first_name"),lit(" "), col("last_name")))
df=df.drop("first_name","last_name")


In [0]:
display(df)

In [0]:
df.write.mode("overwrite").format("delta").save("abfss://silverstg@storageloweretep1.dfs.core.windows.net/buyers")

In [0]:
%sql
CREATE SCHEMA databrickscatalogetep1.silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databrickscatalogetep1.silver.buyerSilver
        USING DELTA 
        LOCATION 'abfss://silverstg@storageloweretep1.dfs.core.windows.net/buyers'


In [0]:
%sql
SELECT * FROM databrickscatalogetep1.silver.buyersilver